In [1]:
import kagglehub, os

img_path = kagglehub.dataset_download("ashwanisingh997/ham10000-images-1229692")
spl_path = kagglehub.dataset_download("ashwanisingh997/ham10000-splits-1229692")
print("IMG:", img_path)
print("SPL:", spl_path)
print("images:", len([f for f in os.listdir(img_path) if f.endswith('.jpg')]))

IMG: /kaggle/input/datasets/ashwanisingh997/ham10000-images-1229692
SPL: /kaggle/input/datasets/ashwanisingh997/ham10000-splits-1229692
images: 10015


In [2]:
import importlib, importlib.metadata as md

try:
    print("before:", md.version("torchao"))
except md.PackageNotFoundError:
    print("before: not installed")

!pip uninstall -y -q torchao

importlib.invalidate_caches()

try:
    print("after:", md.version("torchao"))
except md.PackageNotFoundError:
    print("after: not installed")

import peft.import_utils as piu
if hasattr(piu.is_torchao_available, "cache_clear"):
    piu.is_torchao_available.cache_clear()

ok = piu.is_torchao_available()
print("peft torchao check:", ok)
assert ok is False, "torchao still visible to peft - LoRA would fail after the GAN arm"
print("CELL 2 OK")

before: 0.10.0
after: not installed
peft torchao check: False
CELL 2 OK


In [3]:
import sys, os, shutil, torch, yaml, pandas as pd

CODE = "/kaggle/input/datasets/ashwanisingh997/thesis-code-1229692"
IMG  = "/kaggle/input/datasets/ashwanisingh997/ham10000-images-1229692"
SPL  = "/kaggle/input/datasets/ashwanisingh997/ham10000-splits-1229692"
WORK = "/kaggle/working/repo"

os.makedirs(f"{WORK}/src", exist_ok=True)
os.makedirs(f"{WORK}/config", exist_ok=True)
for f in os.listdir(CODE):
    dst = f"{WORK}/config/{f}" if f.endswith(".yaml") else f"{WORK}/src/{f}"
    shutil.copy(f"{CODE}/{f}", dst)
sys.path.insert(0, WORK)
os.chdir(WORK)

print(torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
src_files = sorted(os.listdir(f"{WORK}/src"))
for need in ["gan.py", "diffusion.py", "budget.py", "data.py"]:
    assert need in src_files, f"MISSING {need}"

tr = pd.read_csv(f"{SPL}/train.csv"); va = pd.read_csv(f"{SPL}/val.csv"); te = pd.read_csv(f"{SPL}/test.csv")
assert (len(tr), len(va), len(te)) == (6981, 1532, 1502), "splits differ from the frozen partition"
assert not (set(tr.lesion_id) & set(te.lesion_id)), "LEAK train/test"
print("splits OK:", len(tr), len(va), len(te))

cfg = yaml.safe_load(open("config/config.yaml"))
print("budget:", cfg["budget"]["gpu_hours_per_arm"], "GPU-h per arm")

from src.run_registry import init_run
root = init_run("run1_cloud", root="/kaggle/working/results", git_hash="5dbc225")
print("ALL CHECKS PASSED")

2.11.0+cu128 | GPU: Tesla T4
splits OK: 6981 1532 1502
budget: 4.0 GPU-h per arm
[registry] run1_cloud (PRIMARY - Kaggle free-tier GPU, matches proposal Table 8.2)
[registry] segment 1 on Tesla T4 | torch 2.11.0+cu128 | cuda 12.8
[registry] writing to /kaggle/working/results/run1_cloud
ALL CHECKS PASSED


In [4]:
from src.utils import set_seed
from src.data import build_image_index, make_generator_loaders

set_seed(42)
index = build_image_index(IMG)
gan_ld, diff_ld, frame = make_generator_loaders(
    f"{SPL}/train.csv", IMG,
    gan_batch=cfg["gan"]["batch_size"],
    diff_size=cfg["data"]["image_size_diffusion"],
    diff_batch=cfg["diffusion"]["batch_size"],
    index=index, num_workers=4)

[data] indexed 10015 images under /kaggle/input/datasets/ashwanisingh997/ham10000-images-1229692
[data] generator source frame: 1526 images across ['akiec', 'bcc', 'df', 'mel', 'vasc']
dx
mel      773
bcc      361
akiec    222
vasc      99
df        71
[data] source data matched across arms: 1526 images, classes ['akiec', 'bcc', 'df', 'mel', 'vasc']


In [5]:
import gc, torch
from src.utils import set_seed
from src.gan import Generator, Critic, train_gan
from src.diffusion import build_lora_unet, load_frozen_components, train_lora
from src.budget import BudgetTracker, assert_same_device, assert_budget_respected

HOURS = cfg["budget"]["gpu_hours_per_arm"]
CKPTS = cfg["budget"]["checkpoint_at_gpu_hours"]
STATE = f"{root}/logs"

print("="*60, "\nARM C — WGAN-GP\n", "="*60, flush=True)
set_seed(42)
G, C = Generator(latent_dim=cfg["gan"]["latent_dim"]), Critic()
with BudgetTracker("gan", "run1_cloud", gpu_hours=HOURS,
                   max_steps=cfg["budget"]["max_optimiser_steps"],
                   checkpoint_at=CKPTS, state_dir=STATE,
                   session_limit_hours=5.0) as bt_gan:
    G, gan_hist = train_gan(G, C, gan_ld, cfg["gan"], bt_gan,
                            save_every_minutes=10.0, out_dir=str(root))

del C
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

print("\n" + "="*60, "\nARM D — LoRA latent diffusion\n", "="*60, flush=True)
set_seed(42)
unet = build_lora_unet(rank=cfg["diffusion"]["lora_rank"], alpha=cfg["diffusion"]["lora_alpha"])
vae, te_enc, tok, sched = load_frozen_components()
with BudgetTracker("diffusion", "run1_cloud", gpu_hours=HOURS,
                   max_steps=cfg["budget"]["max_optimiser_steps"],
                   checkpoint_at=CKPTS, state_dir=STATE,
                   session_limit_hours=5.0) as bt_diff:
    unet, diff_hist = train_lora(unet, vae, te_enc, tok, sched, diff_ld,
                                 cfg["diffusion"], bt_diff,
                                 save_every_minutes=10.0, out_dir=str(root))

assert_same_device("run1_cloud", state_dir=STATE)
assert_budget_respected("run1_cloud", state_dir=STATE)
print("\nBOTH ARMS COMPLETE")

ARM C — WGAN-GP
[budget] arm=gan run=run1_cloud device=Tesla T4 declared=4.00 GPU-h consumed=0.000 segment=1
[budget] checkpoint 1.0h at step 986 (cumulative 1.000 GPU-h)
[budget] checkpoint 2.0h at step 1971 (cumulative 2.001 GPU-h)
[budget] checkpoint 3.0h at step 2956 (cumulative 3.000 GPU-h)
[budget] checkpoint 4.0h at step 3943 (cumulative 4.001 GPU-h)
[budget] segment closed: 4.001 GPU-h. cumulative 4.001 / 4.00 GPU-h, 3943 steps, reason=gpu_hours_exhausted

ARM D — LoRA latent diffusion


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B / 3.44GB            

unet/diffusion_pytorch_model.safetensors: downloading bytes:           |  0.00B            

[lora] trainable 1.59M of 861.1M (0.185%)


... (9 outputs left out)

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

[budget] arm=diffusion run=run1_cloud device=Tesla T4 declared=4.00 GPU-h consumed=0.000 segment=1
[budget] checkpoint 1.0h at step 684 (cumulative 1.001 GPU-h)
[budget] checkpoint 2.0h at step 1367 (cumulative 2.001 GPU-h)
[budget] checkpoint 3.0h at step 2054 (cumulative 3.000 GPU-h)
[budget] checkpoint 4.0h at step 2741 (cumulative 4.001 GPU-h)
[budget] segment closed: 4.001 GPU-h. cumulative 4.001 / 4.00 GPU-h, 2741 steps, reason=gpu_hours_exhausted
[budget] run1_cloud: all segments of all arms on Tesla T4 (2 segment(s)) - control holds.
[budget] run1_cloud budget respected: gan=4.001h, diffusion=4.001h against 4.00h declared.

BOTH ARMS COMPLETE


In [6]:
import shutil
shutil.make_archive("/kaggle/working/run1_generators", "zip", str(root))
print("saved: /kaggle/working/run1_generators.zip")

saved: /kaggle/working/run1_generators.zip
